# Probe runtime Colab cho đồ án Llama sinh mã

Repo public: https://github.com/datkony/llama-code-generation.git

**Mỗi lần mình push code mới, bạn phải đóng tab rồi mở lại notebook, rồi chạy lại cell 2 để `git pull` lấy code mới.** Nếu không, bạn đang chạy code cũ trong bản sao Colab.

Cell chạy trên máy ảo của Google → cell 2 copy/pull repo vào máy ảo. Đây không phải repo thứ hai.

**HF token** được dán vào ô nhập hiện dưới cell 4; token chỉ nằm trong biến môi trường kernel, không ghi vào file. Extension Colab trên VS Code không dùng được `google.colab.userdata`.

Lấy token: chấp nhận license tại https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct rồi tạo token `read` tại https://huggingface.co/settings/tokens.

Chạy 1 → 2 → 3 → 4 → 5. Nếu để trống token, cell 5 tự chạy `--offline` chỉ để validate pipeline; KHÔNG dùng số liệu đó trong báo cáo.

In [1]:
# Colab có sẵn diffusers 0.42.1 đòi huggingface-hub>=1.32, xung đột với transformers 4.x.
# Đồ án không dùng diffusers → gỡ bỏ để pip không báo dependency conflict.
!pip -q uninstall -y diffusers
!pip -q install "transformers>=4.49,<5" "accelerate>=1.3,<2" "huggingface_hub>=0.30,<1" "bitsandbytes>=0.45,<1" gradio

import torch, transformers, huggingface_hub
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__, "| hub", huggingface_hub.__version__)

torch 2.14.1+cpu | cuda False
transformers 4.57.6 | hub 0.36.2


In [ ]:
import os, subprocess


def ask_secret(label):
    print("👉 " + label + " — nhập ở ô bên dưới; KHÔNG dán vào file notebook.")
    value = ""
    try:
        from getpass import getpass
        value = getpass(label + ": ")
    except Exception:
        value = ""
    if not value:
        value = input(label + " (input): ")
    return value.strip()


REPO_URL = "https://github.com/datkony/llama-code-generation.git"
TARGET = "llama-code-generation"
env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"

if os.path.isdir(os.path.join(TARGET, ".git")):
    print("BƯỚC 2: repo đã có → git pull để lấy code mới nhất.")
    subprocess.run(["git", "-C", TARGET, "pull", "--ff-only"], check=True, env=env)
    print("BƯỚC 2: pull xong.")
else:
    print("BƯỚC 2: clone public...")
    try:
        subprocess.run(["git", "clone", "-q", REPO_URL, TARGET], check=True, timeout=180, env=env)
        print("BƯỚC 2: clone public THÀNH CÔNG.")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
        print("BƯỚC 2: clone public THẤT BẠI → repo đang private.")
        print("Chi tiết:", str(exc)[:300])
        token = ask_secret("GitHub PAT (scope repo)")
        if not token:
            raise RuntimeError("Cần GitHub PAT hoặc đổi repo sang public rồi chạy lại cell này.")
        auth_url = REPO_URL.replace("https://", f"https://{token}@")
        try:
            subprocess.run(["git", "clone", "-q", auth_url, TARGET], check=True, timeout=180, env=env)
            print("BƯỚC 2: clone bằng PAT THÀNH CÔNG.")
            subprocess.run(["git", "-C", TARGET, "remote", "set-url", "origin", REPO_URL], check=True, env=env)
            print("BƯỚC 2: đã xóa PAT khỏi .git/config.")
        finally:
            token = ""
            auth_url = ""

os.chdir(TARGET)
print("BƯỚC 2: working dir =", os.getcwd())

Clone public thất bại (repo đang private).


In [ ]:
!python scripts/check_runtime.py

In [ ]:
import os

print("=" * 60)
print("BƯỚC 4: ô nhập HF token sẽ hiện NGAY DƯỚI cell này.")
print("Dán token vào ô đó rồi nhấn Enter. Để trống nếu chưa có token.")
print("Token KHÔNG được ghi vào file notebook.")
print("=" * 60)

token = input("HF token: ").strip()

if not token:
    try:
        from getpass import getpass
        token = getpass("HF token (getpass dự phòng): ").strip()
    except Exception:
        token = ""

if token:
    os.environ["HF_TOKEN"] = token
    print("BƯỚC 4: HF_TOKEN present = True")
else:
    print("BƯỚC 4: HF_TOKEN present = False → bỏ qua smoke endpoint.")
token = ""

if os.environ.get("HF_TOKEN"):
    get_ipython().system("python scripts/smoke_test.py --backend endpoint")
else:
    print("BƯỚC 4: cần HF token + chấp nhận license Llama 3.1 để chạy model thật.")

In [ ]:
import os

if os.environ.get("HF_TOKEN"):
    get_ipython().system("python scripts/run_benchmark.py --variants A,B")
else:
    print("Chưa có HF_TOKEN → chỉ chạy --offline để validate pipeline. KHÔNG báo cáo số liệu này.")
    get_ipython().system("python scripts/run_benchmark.py --offline --variants A,B")
get_ipython().system("python -m unittest discover -s tests -v")